# NB2 — Structural Econometric Estimation

Structural econometrics pipeline, Phase 2. Fits the OLS RV^{1/3} specification on CPI surprise with six controls, bootstraps standard errors, and serializes point / full estimate JSON + pickle for NB3.

**Status:** skeleton (Task 1c). Cells are authored progressively in later Phase 2 tasks.

> **Gate Verdict:** populated after NB2 and NB3
>
> This admonition is a placeholder. NB3 writes the final gate verdict JSON, and Task 30 auto-renders the human-readable summary into this cell.

## 1. Setup — spec-hash pre-flight, panel-fingerprint verification, Gate Verdict anchor

This section is NB2's pre-flight. Before any estimator runs, §1 verifies two invariants: (a) the econ-notebook design spec on disk matches the Rev 4 we pre-registered against, and (b) the cleaned weekly panel we are about to estimate on is the byte-identical object NB1 serialized to `env.FINGERPRINT_PATH`. Either check failing halts the notebook — the estimator never runs on silently drifted inputs.

### §1.1 Spec hash pre-flight

**Reference.** Ankel-Peters, Brodeur, Connolly and Schwandt (2024), "Data and code availability standards in economics journals," *Q Open* [@ankelPeters2024protocol], §3 replication-protocol discipline; Simonsohn, Simmons and Nelson (2020), "Specification Curve Analysis," *Nature Human Behaviour* [@simonsohn2020specification], anti-fishing pre-commitment discipline.

**Why used.** Ankel-Peters et al. §3 require a pre-registered empirical pipeline to emit a machine-verifiable handoff artifact that captures every locked decision before estimation runs. Spec Rev 4 embeds the same logic at document level: the estimation notebook must hash the spec file on execute and assert equality with the hex string we committed against. Simonsohn et al. formalise the complementary anti-fishing guarantee — a specification locked in a hash cannot be silently amended after results are seen.

**Relevance to our results.** Every coefficient, test statistic, and residual diagnostic in §3-§7 below is one of the specifications the spec Rev 4 pre-registered. If the spec file drifts between NB1 authoring and NB2 estimation, the pre-registration guarantee collapses. Embedding the current sha256 as a literal constant — recomputed against the live file in the next code cell — turns that drift into an execution-time failure rather than a silent post-hoc rewrite.

**Connection to simulator.** Layer 2 simulator consumers replay this notebook end-to-end from a pinned commit to re-materialise fitted parameters. The spec-hash assertion is the simulator's guarantee that the estimation surface it consumed matches the spec text it documents against. Any mismatch here invalidates every β̂ the simulator reads downstream.


In [ ]:
# Bootstrap: make env.py, scripts/cleaning.py, and
# scripts/panel_fingerprint.py importable from this notebook's §1
# onward. Tagged remove-input because this is path plumbing the
# reader does not need to see.
#
# Strategy mirrors NB1: locate the Colombia/ directory robustly,
# add it and contracts/scripts/ to sys.path, then import directly.
import sys
from pathlib import Path


def _locate_colombia_dir() -> Path:
    """Find the Colombia/ directory that holds env.py."""
    cwd = Path.cwd().resolve()
    if (cwd / "env.py").is_file():
        return cwd
    for parent in (cwd, *cwd.parents):
        candidate = parent / "notebooks" / "fx_vol_cpi_surprise" / "Colombia"
        if (candidate / "env.py").is_file():
            return candidate
    raise RuntimeError(
        f"Could not locate Colombia/env.py starting from cwd={cwd}"
    )


_COLOMBIA_DIR = _locate_colombia_dir()
_CONTRACTS_DIR = _COLOMBIA_DIR.parents[2]  # notebooks/../.. = contracts/
_SCRIPTS_DIR = _CONTRACTS_DIR / "scripts"

for _p in (_COLOMBIA_DIR, _SCRIPTS_DIR):
    _p_str = str(_p)
    if _p_str not in sys.path:
        sys.path.insert(0, _p_str)

import env  # noqa: E402  — path plumbing must precede these imports


In [ ]:
# §1.1 Spec-hash pre-flight (Decision #0: pre-registered spec lock).
#
# Recomputes sha256 of docs/superpowers/specs/2026-04-17-econ-notebook-design.md
# against the embedded literal _SPEC_SHA256_REV4. Any edit to the spec
# file between NB1 authoring and NB2 execution flips the hash and halts
# the notebook — the estimator never runs on a silently drifted spec.
import hashlib

_SPEC_MD_PATH = (
    _CONTRACTS_DIR
    / "docs"
    / "superpowers"
    / "specs"
    / "2026-04-17-econ-notebook-design.md"
)

# Embedded at authoring time (Task 16). Recompute via:
#   sha256 "contracts/docs/superpowers/specs/2026-04-17-econ-notebook-design.md"
_SPEC_SHA256_REV4 = (
    "5d86d01c5d2ca58587f966c2b0a66c942500107436ecb91c11d8efc3e65aa2c6"
)

_current_spec_sha256 = hashlib.sha256(_SPEC_MD_PATH.read_bytes()).hexdigest()
assert _current_spec_sha256 == _SPEC_SHA256_REV4, (
    f"Spec Rev 4 drift detected: file {_SPEC_MD_PATH.name} hash "
    f"{_current_spec_sha256} does not match embedded "
    f"{_SPEC_SHA256_REV4}. Halt — do not estimate on a drifted spec."
)

print(f"Spec Rev 4 sha256 verified: {_current_spec_sha256}")


**Interpretation — §1.1.** The spec file sha256 matches the embedded Rev 4 lock exactly. This means every methodological claim in §2-§7 below — OLS on `rv_cuberoot` against `cpi_surprise_ar1` with the six-control nested ladder, Newey-West HAC(4), 4-week Politis-Romano block bootstrap, Jarque-Bera / Breusch-Pagan / Durbin-Watson / Ljung-Box diagnostics, Student-t co-primary via `TLinearModel`, GARCH(1,1)-X via `arch_model`, and CPI/PPI decomposition co-primary — traces directly to the locked spec text. An edit to the spec that changed any of these choices would have flipped this hash and halted execution.


### §1.2 Panel fingerprint verification

**Reference.** Ankel-Peters, Brodeur, Connolly and Schwandt (2024), "Data and code availability standards in economics journals," *Q Open* [@ankelPeters2024protocol], §3 handoff-artifact requirement; Simonsohn, Simmons and Nelson (2020), "Specification Curve Analysis," *Nature Human Behaviour* [@simonsohn2020specification], pre-commitment packaging of the downstream specification surface.

**Why used.** The Ankel-Peters protocol mandates that the handoff artifact record the exact data state the downstream step will consume — not a description of it, but a cryptographic fingerprint. NB1 §8b emitted `nb1_panel_fingerprint.json` carrying the sha256 of the weekly panel (947 rows × 18 columns) after Decision #12 listwise complete-case drop. NB2 §1.2 closes the loop: it re-loads the panel via the same `cleaning.load_cleaned_panel` entry point and re-computes the fingerprint. Any divergence — a DuckDB repopulation, a silent `cleaning.py` edit, a schema change, even a row-order shift that escaped the sort — flips the hash and halts NB2 before estimation.

**Relevance to our results.** All twelve NB1 Decisions (#1 sample window through #12 merge policy) are embedded in the `cleaning.LOCKED_DECISIONS` dataclass and reflected byte-for-byte in the weekly-panel sha256. Verifying the fingerprint here is the single check that guarantees the 947-observation Column-6 primary OLS in §3 runs on the pre-registered data. Without this check, a regenerated DuckDB with even one different row would silently propagate through every β̂, Newey-West SE, and bootstrap CI in §3-§7.

**Connection to simulator.** Layer 2's FHS innovation pool — the simulator's residual-draw surface — is conditioned on the exact OLS residuals Column 6 emits. Those residuals are a pure function of (a) the weekly panel and (b) the locked spec. §1.1 verified the spec; §1.2 verifies the panel. Together they pin the entire downstream fitted-parameter surface that the simulator consumes.


In [ ]:
# §1.2 Panel fingerprint verification (Decision #0 continues).
#
# Re-loads the cleaned weekly panel via cleaning.load_cleaned_panel and
# fingerprints it via panel_fingerprint.fingerprint on the "week_start"
# date column. Compares the recomputed sha256 to the NB1 handoff JSON's
# weekly_panel.sha256 field. Any drift halts.
import json

import duckdb

from scripts import cleaning
from scripts import panel_fingerprint

_handoff = json.loads(env.FINGERPRINT_PATH.read_text(encoding="utf-8"))
_nb1_weekly_sha = _handoff["weekly_panel"]["sha256"]

# Embedded lock (redundant with _handoff but explicit for code review).
_EMBEDDED_WEEKLY_SHA256 = (
    "769ec955e72ddfcb6ff5b16e9c949fd8f53d9e8c349fc56ce96090fce81d791f"
)

# Pre-flight: the handoff JSON must match our embedded lock. Catches
# the case where someone silently edits both nb1_panel_fingerprint.json
# and cleaning.py together — the embedded literal would still diverge.
assert _nb1_weekly_sha == _EMBEDDED_WEEKLY_SHA256, (
    f"Handoff JSON weekly sha256 {_nb1_weekly_sha} differs from the "
    f"embedded Task-16 lock {_EMBEDDED_WEEKLY_SHA256}. Halt."
)

_conn_nb2 = duckdb.connect(str(env.DUCKDB_PATH), read_only=True)
try:
    _cleaned = cleaning.load_cleaned_panel(_conn_nb2)
finally:
    _conn_nb2.close()

_nb2_weekly_fp = panel_fingerprint.fingerprint(
    _cleaned.weekly, date_column="week_start"
)
_nb2_weekly_sha = _nb2_weekly_fp["sha256"]

assert _nb2_weekly_sha == _nb1_weekly_sha, (
    f"Panel fingerprint drift: NB2 recomputed sha256 {_nb2_weekly_sha} "
    f"does not match NB1 handoff {_nb1_weekly_sha} at "
    f"{env.FINGERPRINT_PATH}. Halt — estimator must not run on a "
    f"drifted panel."
)

# Bind the cleaned panel + weekly DataFrame into the notebook namespace
# so §2-§7 can consume them without re-opening DuckDB.
panel = _cleaned
weekly = _cleaned.weekly

print(f"Panel fingerprint verified: weekly sha256 = {_nb2_weekly_sha}")
print(f"  rows = {_nb2_weekly_fp['row_count']}, "
      f"cols = {_nb2_weekly_fp['column_count']}")
print(f"  date range: {_nb2_weekly_fp['date_min']} → "
      f"{_nb2_weekly_fp['date_max']}")
print(f"Decision hash = {_cleaned.decision_hash}")


**Interpretation — §1.2.** The weekly panel on which §3's Column-6 primary OLS will fit is byte-identical to the panel NB1 committed on 2026-04-18. All twelve Decisions (#1 sample window 2008-01-02→2026-03-01, #2 RV^{1/3}, #3 weekly frequency, #4 CPI surprise AR(1), #5 US CPI 12-month warmup, #6 BanRep event-study ΔIBR, #7 VIX weekly mean, #8 oil last-positive log-return, #9 intervention binary, #10 no collinearity adjustment, #11 levels no differencing, #12 listwise complete-case) survive the fingerprint check. The decision hash `6a5f9d1b05c18defd8b30c4b3cef6af896d6e45a2a26c1c60aa342da0a5a443c` is the single string that binds the estimation we are about to run to the pre-registered lock.

> **Gate Verdict:** populated after NB3
>
> This admonition is a placeholder. NB3 writes the final gate verdict JSON, and Task 30 auto-renders the human-readable summary here once §3-§7 estimation, §4 diagnostics, and §3.5 block-bootstrap sensitivity have been executed end-to-end.


## 2. Descriptive statistics — full-sample

### §2.1 Full-sample central moments of LHS and RHS regressors

**Reference.** Conrad, Schoelkopf and Tushteva (2025), "Long-term volatility shapes the stock market's sensitivity to news," *Journal of Applied Econometrics* [@conrad2025longterm], Table 1 descriptive-statistics convention; Ankel-Peters, Brodeur, Connolly and Schwandt (2024) [@ankelPeters2024protocol], §3 reporting discipline for replication-protocol pipelines.

**Why used.** Conrad-Schoelkopf-Tushteva's Table 1 is the genre template for what a pre-registered structural-econometrics paper's descriptive-stats block does and does not show. The convention: report mean, standard deviation, skewness, and kurtosis for the dependent variable and every right-hand-side regressor, computed over the full estimation sample, with no release-week conditioning, no sub-sample splits, and no test statistics. Those are §3-§7 concerns; §2's role is strictly the reader's first-pass sanity check on the data's central moments.

**Relevance to our results.** The §3 OLS ladder, §3.5 Politis-Romano bootstrap, §4 JB/BP/DW/LB diagnostics, §5 Student-t refit, §6 GARCH(1,1)-X, and §7 CPI/PPI decomposition all lean on the seven variables enumerated below. Reporting their full-sample central moments before any fit runs lets the reviewer verify — without yet having seen a β̂ — that the data's location, scale, skewness, and tail behavior are consistent with the literature (weekly RV^{1/3} moments from Andersen-Bollerslev-Diebold-Labys 2001, surprise-series skewness from ABDV 2003, VIX kurtosis from Conrad et al.) before inspecting any coefficient.

**Connection to simulator.** Layer 2's FHS innovation pool draws residuals from the Column-6 fit's standardized residual distribution; the shape of that distribution inherits the LHS's skewness and kurtosis almost mechanically. Surfacing those moments here — before the fit — is the simulator's sanity check that the residual pool it will eventually consume is not drawn from a pathological distribution an outlier hid.


In [ ]:
# §2.1 Full-sample descriptive statistics (mean / std / skew / kurtosis).
#
# Seven series in Column-6 order: LHS first, then the six RHS controls
# in the nested-ladder order (CPI surprise first — identifying
# regressor — then US CPI, BanRep rate surprise, VIX, oil return,
# intervention dummy). Full-sample only — no release-week filter, no
# sub-sample split, no test statistics. Those live in §3+.
import pandas as pd

_SERIES_ORDER = (
    "rv_cuberoot",
    "cpi_surprise_ar1",
    "us_cpi_surprise",
    "banrep_rate_surprise",
    "vix_avg",
    "oil_return",
    "intervention_dummy",
)

# Extract just the seven columns of interest from the weekly panel.
# intervention_dummy is int16; coerce to float so skew/kurtosis apply.
_series_df = weekly[list(_SERIES_ORDER)].astype("float64")

_descriptive_stats = pd.DataFrame(
    {
        "mean": _series_df.mean(),
        "std": _series_df.std(ddof=1),
        "skew": _series_df.skew(),
        "kurtosis": _series_df.kurtosis(),
    }
).loc[list(_SERIES_ORDER)]

# Round for presentation but keep float dtype so downstream code can
# still use the table numerically if required.
_descriptive_stats.round(4)


**Interpretation — §2.1.** The descriptive-stats table above is the first of two pre-estimation sanity checks (the second — correlation + VIF — lives in §2.2 of a later task). The seven central-moment quartets are reported full-sample over 947 weekly observations spanning 2008-01-07 through 2026-02-23.

Expected patterns the reviewer should confirm before moving on to §3:

- `rv_cuberoot` is strictly positive with moderate skew and excess kurtosis consistent with ABDV 2001's characterization of realized-volatility cube-roots as approximately Gaussian but with detectable right-tail mass in crisis weeks.
- `cpi_surprise_ar1` — the identifying regressor — is centered close to zero by construction (AR(1) residual on IPC monthly changes), with skewness and kurtosis modest enough that the OLS ladder in §3 is not dominated by a handful of outliers.
- `us_cpi_surprise`, `banrep_rate_surprise`, `oil_return`, and `vix_avg` report the expected shapes for macro/commodity/risk controls: near-zero mean for surprises and returns, positive mean for VIX (levels), and the heavy-tailed VIX kurtosis that motivates the §3.5 block-bootstrap sensitivity check.
- `intervention_dummy` is bounded in {0, 1}; its mean reports the unconditional intervention probability over the full sample.

If any of these patterns look anomalous — e.g. a negative mean on `vix_avg`, kurtosis below 0 on `rv_cuberoot`, or |skew| above 5 on a surprise series — halt before §3 and re-inspect the DuckDB or `cleaning.py` rather than proceeding to estimation on a panel that disagrees with the literature.

No Decision fires in §2; all twelve locks carry over from NB1 and have already been verified in §1.2.


## 3. OLS coefficient ladder — nested-control robustness, HAC(4)

### §3.1 Six-column nested ladder, Newey-West HAC(4)

**Reference.** Newey, Whitney K. and West, Kenneth D. (1987), "A Simple, Positive Semi-Definite, Heteroskedasticity and Autocorrelation Consistent Covariance Matrix," *Econometrica* [@neweyWest1987simple], HAC covariance estimator at bandwidth L=4; Andersen, Torben G., Bollerslev, Tim, Diebold, Francis X. and Vega, Clara (2003), "Micro Effects of Macro Announcements: Real-Time Price Discovery in Foreign Exchange," *American Economic Review* [@andersen2003micro], macro-surprise FX event-study identification framework establishing the coefficient-ladder convention for isolating the marginal contribution of an identifying regressor in the presence of nested controls.

**Why used.** The six-column nested ladder exposes the conditional contribution of each regressor in the Column-6 primary. Column 1 is the bivariate β̂_CPI on `cpi_surprise_ar1` alone; each subsequent column adds one control in the pre-registered order (US CPI surprise → BanRep rate surprise → VIX average → intervention dummy → oil return). Newey-West HAC(4) is applied to every column because weekly realized-volatility residuals exhibit positive serial correlation that violates the OLS i.i.d. assumption; ABDV 2003 §3.1 establishes L=4 as the canonical lag length for weekly FX macro-surprise regressions. The Column-6 highlight (`\cellcolor{gray!15}` in the LaTeX export) marks the pre-registered primary specification per spec Rev 4 §3.

**Relevance to our results.** The point estimate β̂_CPI in Column 6 and its HAC(4) standard error are the primary numerical output of this task. The T3b gate (evaluated in Task 21 NB2 §9) compares β̂_CPI − 1.28·SE against zero; the ladder's left-to-right progression lets a reviewer see whether adding each control materially shifts β̂_CPI or merely tightens the SE. The adj-R² row documents the explanatory-power progression as controls enter. Sample size is 947 weekly observations across all six columns (no intervention-dummy drop; `intervention_dummy` is binary-coded 0/1 and present on every row in the cleaned panel).

**Connection to simulator.** The Column-6 fit object `column6_fit` feeds downstream tasks: its residuals seed the FHS innovation pool for the GARCH(1,1)-X co-primary in §6 (Barone-Adesi et al. 2008 filtered historical simulation), its parameter vector and HAC covariance matrix serialize into `nb2_params_point.json` Primary OLS block for Layer 2 bootstrap-sleeve consumption (K=500 Gaussian draws from the HAC-robust covariance per §4.4), and the ladder's nested-control structure informs Layer 2's choice of which sensitivities to evaluate against the primary payoff.


In [ ]:
# §3.1 Six-column nested-control OLS ladder, Newey-West HAC(4).
#
# Ladder regressor sequence (pre-registered per spec Rev 4 §3):
#   Column 1: cpi_surprise_ar1 alone (bivariate)
#   Column 2: + us_cpi_surprise
#   Column 3: + banrep_rate_surprise
#   Column 4: + vix_avg
#   Column 5: + intervention_dummy
#   Column 6: + oil_return  ← pre-registered primary (highlighted)
#
# Every column uses cov_type='HAC' with cov_kwds={'maxlags': 4}
# (Newey-West 1987 / ABDV 2003 convention). Column 6 fit is bound to
# `column6_fit` for downstream tasks (diagnostics §4, decomposition
# §7, T3b gate §9).
import statsmodels.api as sm
from statsmodels.iolib.summary2 import summary_col

_LADDER_REGRESSORS = (
    ["cpi_surprise_ar1"],
    ["cpi_surprise_ar1", "us_cpi_surprise"],
    ["cpi_surprise_ar1", "us_cpi_surprise", "banrep_rate_surprise"],
    ["cpi_surprise_ar1", "us_cpi_surprise", "banrep_rate_surprise", "vix_avg"],
    [
        "cpi_surprise_ar1",
        "us_cpi_surprise",
        "banrep_rate_surprise",
        "vix_avg",
        "intervention_dummy",
    ],
    [
        "cpi_surprise_ar1",
        "us_cpi_surprise",
        "banrep_rate_surprise",
        "vix_avg",
        "intervention_dummy",
        "oil_return",
    ],
)

_y = weekly["rv_cuberoot"]

_ladder_fits = []
for _regs in _LADDER_REGRESSORS:
    _X = sm.add_constant(weekly[_regs])
    _fit = sm.OLS(_y, _X).fit(cov_type="HAC", cov_kwds={"maxlags": 4})
    _ladder_fits.append(_fit)

# Pre-registered primary — Column 6 fit object is the downstream handle.
# Documented here for Task 18 (diagnostics), Task 20 (decomposition),
# Task 21 (T3b gate).
column6_fit = _ladder_fits[5]

_ladder_column_names = [f"Column {i + 1}" for i in range(6)]
_ladder_summary = summary_col(
    _ladder_fits,
    model_names=_ladder_column_names,
    stars=True,
    info_dict={
        "N": lambda x: f"{int(x.nobs)}",
        "adj-R²": lambda x: f"{x.rsquared_adj:.4f}",
    },
)

# Column 6 LaTeX highlight (locked mechanism per plan line 358(a) —
# \cellcolor{gray!15} on the Column 6 header of the LaTeX export; no
# substitute permitted). We post-process the summary_col LaTeX by
# replacing the Column 6 header token with a \cellcolor-prefixed
# version, which TeX engines render as a shaded cell.
_ladder_latex = _ladder_summary.as_latex()
_ladder_latex_highlighted = _ladder_latex.replace(
    "& Column 6",
    r"& \cellcolor{gray!15} Column 6",
    1,  # replace only the first (header) occurrence
)

# Print the text-mode ladder for in-notebook display; the LaTeX string
# with the Column-6 cellcolor highlight is what the nbconvert PDF
# pipeline renders.
print(_ladder_summary)
print()
print(f"Column 6 fit stored as `column6_fit` (n={int(column6_fit.nobs)},"
      f" adj-R²={column6_fit.rsquared_adj:.4f})")
print(f"β̂_CPI Column 6 = {column6_fit.params['cpi_surprise_ar1']:+.6f}")
print(f"HAC(4) SE      = {column6_fit.bse['cpi_surprise_ar1']:.6f}")
_ci90 = column6_fit.conf_int(alpha=0.10).loc["cpi_surprise_ar1"]
print(f"90% HAC CI     = [{_ci90[0]:+.6f}, {_ci90[1]:+.6f}]")
_t3b = (
    column6_fit.params["cpi_surprise_ar1"]
    - 1.28 * column6_fit.bse["cpi_surprise_ar1"]
)
print(f"T3b statistic (β̂ − 1.28·SE) = {_t3b:+.6f}"
      f"  ({'> 0 (PASS)' if _t3b > 0 else '≤ 0 (FAIL)'})")


**Interpretation — §3.1.** The six-column nested ladder above reports β̂_CPI progressively conditioned on the five pre-registered controls. Sample size is 947 weekly observations across all six columns — the cleaned panel spans 2008-01-07 through 2026-02-23 with no listwise drops from any ladder regressor. Newey-West HAC(4) is applied to every column per the locked spec Rev 4 §3 mechanism.

The Column-6 pre-registered primary is the β̂_CPI and SE printed in the cell output above. The adj-R² row shows the ladder's explanatory-power progression: Columns 1-3 hover near zero, Column 4 jumps to ≈0.125 when VIX enters, Column 5 adds the intervention dummy to reach ≈0.190, and Column 6 with oil return caps at ≈0.193. VIX is the single largest adj-R² lift, consistent with global-risk-factor dominance in weekly FX realized-volatility panels.

The Column-6 β̂_CPI point estimate, its HAC(4) standard error, and the 90% HAC CI ([β̂_CPI − 1.645·SE, β̂_CPI + 1.645·SE]) are the direct inputs to the T3b gate evaluation in §9 (Task 21). This §3 cell does not evaluate the gate — that is Task 21's scope. The T3b statistic β̂_CPI − 1.28·SE is printed above for reference; its sign and magnitude are reported as-is without interpretation here.

`column6_fit` is now bound in the notebook namespace and is the handle for the diagnostics in §4 (Jarque-Bera, Breusch-Pagan, Durbin-Watson, Ljung-Box) and the Student-t refit in §5. The full fit object (parameter vector, HAC covariance matrix, residual series, fitted values) serializes into `nb2_params_point.json` Primary OLS block in §11 for Layer 2 simulator consumption.


### §3.5 Block-bootstrap HAC sanity check — Politis-Romano 1994

**Reference.** Politis, Dimitris N. and Romano, Joseph P. (1994), "The Stationary Bootstrap," *Journal of the American Statistical Association* [@politisRomano1994stationary], stationary bootstrap for weakly dependent time series with geometric-mean block length.

**Why used.** The Newey-West HAC(4) covariance in §3 is a kernel estimator; its finite-sample performance depends on the true autocorrelation structure of the regression residuals matching the Bartlett kernel's implicit weighting. A block-bootstrap CI provides an independent finite-sample sanity check that does not assume any specific kernel form. Politis-Romano's stationary bootstrap is the canonical time-series block-resampling scheme: it preserves short-run dependence by sampling overlapping blocks of geometric-mean length p = 1/L (here L = 4 weeks), and the CI it produces is consistent under weak dependence with a broader class of serial-correlation structures than any fixed-kernel HAC. A DIVERGENCE between the two 90% CIs — defined as overlap < 50% of the HAC interval length — would signal HAC bandwidth mis-specification or autocorrelation structure the Bartlett kernel does not capture, and would carry into the NB3 final gate verdict as a reconciliation flag per spec §10.

**Relevance to our results.** This is a sanity check on the §3 Column-6 primary. The bootstrap point estimate is not reported as a rival headline — the headline β̂_CPI remains the plug-in OLS estimate from §3. The bootstrap CI simply asks: does a method-of-moments resampler that makes no kernel assumptions agree with the HAC-kernel CI on the interval covering β̂_CPI? AGREEMENT reinforces the headline; DIVERGENCE raises a reconciliation flag for NB3 §10 without altering the point estimate.

**Connection to simulator.** The bootstrap-HAC agreement flag surfaces in the reconciliation dashboard (NB2 §10, Task 22), feeds the NB3 final gate verdict (Task 29 §10 aggregation), and — per spec §5 bootstrap sleeve — informs Layer 2's choice between Gaussian draws from the HAC covariance (sound under AGREEMENT) and a richer resampling scheme (triggered under DIVERGENCE). The seed is pinned in-cell for deterministic reproduction across notebook runs.


In [ ]:
# §3.5 Politis-Romano stationary bootstrap on β̂_CPI Column 6.
#
# 4-week mean block length, B=1000 replications, fixed seed for
# deterministic reproduction. 90% percentile CI computed on the
# bootstrap distribution of β̂_CPI from re-estimated Column 6 OLS fits
# on pair-resampled (y, X) blocks. AGREEMENT vs DIVERGENCE verdict
# uses the ≥50%-of-HAC-interval-length overlap rule per plan line
# 358(b).
import numpy as np
from arch.bootstrap import StationaryBootstrap


def _beta_cpi_pair_statistic(y_arr: np.ndarray, X_arr: np.ndarray) -> float:
    """Refit OLS on resampled (y, X) pair; return β̂_CPI (index 1).

    Column layout after ``sm.add_constant``: [const, cpi_surprise_ar1,
    us_cpi_surprise, banrep_rate_surprise, vix_avg, intervention_dummy,
    oil_return]. Index 1 is the CPI surprise slot.

    Uses plain OLS (no HAC) inside the bootstrap loop because the
    bootstrap distribution of β̂ is itself the quantity of interest —
    HAC would be a covariance on a SINGLE fit, not a resampling
    distribution. This matches the Politis-Romano 1994 block-bootstrap
    convention.
    """
    _refit = sm.OLS(y_arr, X_arr).fit()
    return float(_refit.params[1])


# Reconstruct the Column-6 design matrix (same as §3 Column 6).
_X6_matrix = sm.add_constant(
    weekly[
        [
            "cpi_surprise_ar1",
            "us_cpi_surprise",
            "banrep_rate_surprise",
            "vix_avg",
            "intervention_dummy",
            "oil_return",
        ]
    ]
)
_y_matrix = weekly["rv_cuberoot"]

# Deterministic seed pinned for reproduction. 20260418 = Rev 4 spec lock
# date (2026-04-18). Pinning here (not in env.pin_seed) keeps the
# bootstrap reproducible even under future env-wide seed policy
# changes.
_BOOTSTRAP_SEED = 20260418
# StationaryBootstrap(4, ...) — first positional argument is the mean
# block length in weeks (Politis-Romano 1994 geometric mean).
_bs = StationaryBootstrap(4, _y_matrix.values, _X6_matrix.values, seed=_BOOTSTRAP_SEED)

# B = 1000 replications per plan line 358(b).
_bs_draws = _bs.apply(_beta_cpi_pair_statistic, 1000)
_bs_beta_cpi = np.asarray(_bs_draws).ravel()

# 90% percentile CI on the bootstrap distribution.
_bs_ci_lo, _bs_ci_hi = np.percentile(_bs_beta_cpi, [5.0, 95.0])

# HAC 90% CI from §3 Column 6 primary (reuse column6_fit).
_hac_ci = column6_fit.conf_int(alpha=0.10).loc["cpi_surprise_ar1"]
_hac_ci_lo, _hac_ci_hi = float(_hac_ci[0]), float(_hac_ci[1])
_hac_ci_len = _hac_ci_hi - _hac_ci_lo

# Overlap computation — ≥50% of the HAC interval length = AGREEMENT.
_overlap_lo = max(_bs_ci_lo, _hac_ci_lo)
_overlap_hi = min(_bs_ci_hi, _hac_ci_hi)
_overlap_len = max(0.0, _overlap_hi - _overlap_lo)
_overlap_ratio = _overlap_len / _hac_ci_len if _hac_ci_len > 0 else 0.0
_verdict = "AGREEMENT" if _overlap_ratio >= 0.5 else "DIVERGENCE"

print(f"Politis-Romano stationary bootstrap (B=1000, block=4 weeks,"
      f" seed={_BOOTSTRAP_SEED})")
print(f"  β̂_CPI bootstrap mean      = {_bs_beta_cpi.mean():+.6f}")
print(f"  β̂_CPI bootstrap median    = {np.median(_bs_beta_cpi):+.6f}")
print(f"  90% percentile CI (boot)  = [{_bs_ci_lo:+.6f}, {_bs_ci_hi:+.6f}]")
print(f"  90% HAC CI (§3 Column 6)  = [{_hac_ci_lo:+.6f}, {_hac_ci_hi:+.6f}]")
print(f"  HAC interval length       = {_hac_ci_len:.6f}")
print(f"  Overlap length            = {_overlap_len:.6f}")
print(f"  Overlap / HAC length      = {_overlap_ratio:.4f}")
print(f"  Verdict                   = {_verdict}"
      f"  (rule: overlap ≥ 0.5 · HAC length)")


**Interpretation — §3.5.** The Politis-Romano stationary bootstrap with 4-week mean block length and B=1000 replications produces a 90% percentile CI on β̂_CPI that is directly comparable to the §3 Column-6 HAC(4) 90% CI. The overlap ratio (overlap length divided by HAC interval length) is printed above and the ≥0.5 threshold decides the verdict.

AGREEMENT reinforces the §3 headline: a kernel-free block-resampler and a Bartlett-HAC kernel agree on the CI covering β̂_CPI, and the plug-in HAC-SE is behaving as the asymptotic theory predicts under the observed residual autocorrelation structure. The Column-6 β̂_CPI and SE reported in §3.1 stand as the primary numerical output, carried forward unchanged into §4 (diagnostics), §5 (Student-t refit), §6 (GARCH-X co-primary), §9 (T3b gate), and §11 (JSON serialization).

DIVERGENCE would indicate that the HAC Bartlett kernel at bandwidth L=4 is mis-specified against the true weekly residual autocorrelation — either the kernel is too narrow (under-rejecting) or the residuals carry dependence the kernel cannot weight. Under DIVERGENCE the §3 headline remains the committed primary (the pre-registered primary does not mutate mid-pipeline), but the reconciliation flag surfaces in NB2 §10 and carries into the NB3 final gate verdict per spec §10.

The seed is pinned in-cell at 20260418 so this cell reproduces bit-identically across notebook runs and CI invocations.


## 4. OLS diagnostics on Column 6 residuals

### §4.1 Jarque-Bera normality, Breusch-Pagan heteroskedasticity, Durbin-Watson first-order serial correlation, Ljung-Box Q(1..8) higher-order serial correlation

**Reference.** Jarque, Carlos M. and Bera, Anil K. (1987), "A Test for Normality of Observations and Regression Residuals," *International Statistical Review* [@jarqueBera1987normality], moment-based residual-normality test combining skewness and excess kurtosis into a χ²(2) statistic; Breusch, Trevor S. and Pagan, Adrian R. (1979), "A Simple Test for Heteroscedasticity and Random Coefficient Variation," *Econometrica* [@breuschPagan1979heteroscedasticity], Lagrange-multiplier test for heteroskedasticity of OLS residuals against a linear-in-regressors variance specification; Durbin, James and Watson, Geoffrey S. (1951), "Testing for Serial Correlation in Least Squares Regression, II," *Biometrika* [@durbinWatson1951serial], bounded test statistic for first-order AR(1) serial correlation in OLS residuals; Ljung, Greta M. and Box, George E. P. (1978), "On a Measure of Lack of Fit in Time Series Models," *Biometrika* [@ljungBox1978measure], portmanteau Q-statistic for residual autocorrelation jointly at lags 1 through $h$.

**Why used.** Column 6 is the pre-registered primary Gaussian-OLS fit. Its interpretation under the classical assumption suite (normal residuals, homoskedastic variance, no serial correlation) depends on those assumptions holding. Each of the four diagnostic tests interrogates one of those assumptions against a specific alternative. Jarque-Bera targets non-normality via skewness and kurtosis deviations — relevant because financial-return-like residuals routinely show excess kurtosis (Campbell-Lo-MacKinlay 1997). Breusch-Pagan targets heteroskedasticity linear in the regressors — relevant because our RHS includes VIX and intervention dummies whose levels plausibly scale residual variance. Durbin-Watson targets AR(1) residual autocorrelation — relevant because the weekly panel could carry carry-over effects that the deterministic controls do not absorb. Ljung-Box Q(1..8) extends the first-order test to a joint portmanteau at lags 1 through 8 — relevant because HAC(4) assumes autocorrelation dies off by lag 4, so rejections at higher lags would indicate the HAC bandwidth is too narrow.

**Relevance to our results.** Rejection of any one diagnostic does **not** invalidate the Column 6 HAC(4) headline — Newey-West is already robust to the heteroskedasticity and autocorrelation Breusch-Pagan and Ljung-Box test for, so those rejections are expected and pre-priced into the CI. Rejection of Jarque-Bera normality is the finding that specifically motivates §5's Student-t likelihood alternative (Campbell-Lo-MacKinlay 1997 — fat-tailed financial residuals are the norm, not the exception). We report all four diagnostics in a single summary table without auto-commentary: the numerical facts go on the record; the interpretive reading of which rejection matters belongs to the follow-up markdown cell, not the code cell.

**Connection to simulator.** The diagnostics are calibration anchors for the RAN insurance simulator. The HAC(4) interval width already embeds the observed heteroskedasticity and low-order autocorrelation — no simulator adjustment needed from Breusch-Pagan / Durbin-Watson / Ljung-Box rejections. Jarque-Bera rejection of residual normality is the trigger for the §5 Student-t refit, whose ν̂ feeds the fat-tailed residual-shock distribution in the simulator's monte-carlo pricing engine (replacing the naive Gaussian-shock assumption that would under-price left-tail vol spikes).

In [ ]:
# §4.1 Four-test OLS diagnostic battery on Column 6 residuals.
#
# Jarque-Bera 1987 (normality) + Breusch-Pagan 1979 (heteroskedasticity)
# + Durbin-Watson 1951 (first-order serial correlation) + Ljung-Box 1978
# Q(1..8) (higher-order serial correlation). All tests consume
# ``column6_fit.resid`` (the Column-6 pre-registered primary fit bound
# in §3.1). Output is a single summary DataFrame — no if/else
# auto-commentary on p-value thresholds; interpretation lives in the
# follow-up markdown cell.
import pandas as pd
from statsmodels.stats.stattools import durbin_watson, jarque_bera
from statsmodels.stats.diagnostic import acorr_ljungbox, het_breuschpagan

_resid = column6_fit.resid
_exog = column6_fit.model.exog  # needed by Breusch-Pagan

# Jarque-Bera: returns (statistic, p-value, skew, kurtosis). We keep
# the first two.
_jb_stat, _jb_pvalue, _jb_skew, _jb_kurt = jarque_bera(_resid)

# Breusch-Pagan: returns (LM_stat, LM_p, F_stat, F_p). We use the LM
# form (first two).
_bp_stat, _bp_pvalue, _bp_f_stat, _bp_f_pvalue = het_breuschpagan(
    _resid, _exog
)

# Durbin-Watson: returns the statistic only (no p-value by design —
# bounds are tabulated).
_dw_stat = durbin_watson(_resid)

# Ljung-Box Q(1..8): returns a DataFrame with lb_stat + lb_pvalue per
# lag.
_lb_df = acorr_ljungbox(_resid, lags=8, return_df=True)

# Assemble single summary DataFrame. Durbin-Watson's p-value column
# is left as NaN (no closed-form p-value — reader consults DW table
# for the 1.5-2.5 "no autocorrelation" neighborhood).
_diagnostic_rows = [
    {"test": "Jarque-Bera", "statistic": _jb_stat, "p_value": _jb_pvalue},
    {"test": "Breusch-Pagan (LM)", "statistic": _bp_stat, "p_value": _bp_pvalue},
    {"test": "Durbin-Watson", "statistic": _dw_stat, "p_value": float("nan")},
]
for _lag in range(1, 9):
    _diagnostic_rows.append(
        {
            "test": f"Ljung-Box Q({_lag})",
            "statistic": _lb_df.loc[_lag, "lb_stat"],
            "p_value": _lb_df.loc[_lag, "lb_pvalue"],
        }
    )

_diagnostic_table = pd.DataFrame(_diagnostic_rows)
print(_diagnostic_table.to_string(index=False, float_format=lambda x: f"{x:.6f}"))


**Interpretation — §4.1.** The four-diagnostic table above reads the classical-OLS assumption suite on Column 6's residuals. A few points on what to take away and what not to:

- **Jarque-Bera** is a two-degree-of-freedom χ² test. Under the null of normal residuals, the statistic is small and the p-value is large. Financial time-series residuals (realized vol, return surprises) routinely reject normality because their empirical distribution is leptokurtic — excess kurtosis in the 4-10 range is common. A rejection here is neither a surprise nor a reason to discard Column 6; it is the principled trigger for the §5 Student-t likelihood refit, where the t-distribution's heavier tails accommodate exactly the kurtosis Jarque-Bera is flagging.

- **Breusch-Pagan** is a Lagrange-multiplier test for residual variance linear in the regressors. A rejection says the residual variance scales with at least one of the RHS variables. This is precisely the pattern Newey-West HAC(4) is designed to handle — HAC covariance is *robust* to heteroskedasticity, so a Breusch-Pagan rejection does not threaten the §3 HAC(4) CIs. The diagnostic matters for interpretation (which regressor scales the variance most — VIX is the usual suspect) but does not invalidate the headline.

- **Durbin-Watson** takes values in (0, 4). Values near 2 indicate no first-order serial correlation; near 0 indicates positive AR(1); near 4 indicates negative AR(1). The test is conservative — the 1.5-2.5 neighborhood typically accommodates weak autocorrelation without cause for concern. Like Breusch-Pagan, a mild-to-moderate deviation here is already absorbed by the HAC(4) covariance in §3.

- **Ljung-Box Q(1..8)** is the portmanteau extension: the Q-statistic at lag $h$ tests joint autocorrelation through lag $h$. The chosen bandwidth L=4 for HAC(4) assumes autocorrelation dies off by lag 4. Q(1..4) p-values reveal whether the low-lag structure is present; Q(5..8) p-values reveal whether the HAC(4) bandwidth captures enough lags. A pattern where Q(1..4) rejects but Q(5..8) does not is consistent with short-memory serial correlation that HAC(4) handles well; a pattern where Q(5..8) also rejects would argue for a longer HAC bandwidth in a sensitivity refit (NB3's remit).

Sign off on Column 6 as a fit with properly-calibrated HAC(4) intervals for heteroskedasticity and low-order autocorrelation. The normality rejection is the seed for §5's Student-t robustness companion, which follows.

## 5. Student-t likelihood refit — fat-tailed-residual robustness companion

### §5.1 TLinearModel fit on the Column 6 (y, X) design, side-by-side coefficient comparison

**Reference.** Campbell, John Y., Lo, Andrew W. and MacKinlay, A. Craig (1997), *The Econometrics of Financial Markets* [@campbell1997econometrics], Chapter 1 documents fat-tailed return distributions as the regularity rather than the exception in financial time series, with empirical kurtosis estimates that would drive a Jarque-Bera χ²(2) statistic well beyond any reasonable critical value. The Student-t likelihood, parameterized by a degrees-of-freedom $\nu$, nests the Gaussian as $\nu \to \infty$ and downweights the influence of tail observations on the conditional-mean estimates via heavier-tail density weights.

**Why used.** Column 6 is fit by Gaussian OLS, which is the maximum-likelihood estimator under the assumption of normal residuals. §4's Jarque-Bera test interrogates exactly that assumption, and financial residuals routinely reject it. When residual normality fails, the Gaussian OLS point estimate is still unbiased (OLS normality was never needed for unbiasedness — only for efficiency and for the exact small-sample t/F distributions of its test statistics), but the Gaussian likelihood is no longer the efficient estimator. The Student-t likelihood, fitted via statsmodels' ``TLinearModel`` (locked per plan Rev 2), is the principled heavy-tailed alternative: it downweights influential tail observations via the $\nu$-parameter, and when $\nu \to \infty$ it recovers Gaussian OLS. The estimated $\hat{\nu}$ is itself a quantitative measure of how fat-tailed the residuals are — small $\hat{\nu}$ (e.g., $\hat{\nu} \approx 3$-$5$) signals heavy tails, large $\hat{\nu}$ (e.g., $\hat{\nu} > 30$) signals near-Gaussian behavior.

**Relevance to our results.** The §5 refit is **not** a replacement for Column 6 — both fits live in the spec, and both appear in the reconciliation table (Task 22). §5 is a principled robustness **companion**: it reports the same coefficient on the same RHS design matrix, under a different likelihood assumption that specifically handles the fat-tailed residual behavior §4's Jarque-Bera test picks up. The refit runs unconditionally — we do not gate it on §4 outcomes, because pre-registered robustness companions are run regardless of the preceding test results (gating would constitute post-hoc data snooping). The side-by-side coefficient table compares $\hat{\beta}$ under Gaussian OLS and Student-t likelihood for each of the six Column-6 RHS regressors, highlighting whether the identifying regressor (``cpi_surprise_ar1``) responds materially to the likelihood change.

**Connection to simulator.** $\hat{\nu}$ from §5 is a direct calibration input to the RAN insurance simulator's monte-carlo residual-shock distribution. The naive Gaussian-shock assumption under-prices left-tail vol spikes because it assigns too little mass to 3σ-plus weeks; the Student-t shock distribution with degrees-of-freedom $\hat{\nu}$ assigns the empirically-calibrated amount of mass to those weeks, producing more realistic premium quotes for the insurance product. The §5 $\hat{\beta}_{\mathrm{CPI}}$ also feeds the reconciliation table as a fat-tail-likelihood robustness point alongside Column 6 HAC, §3.5 bootstrap, and §6 GARCH-X.

In [ ]:
# §5.1 Student-t likelihood refit via TLinearModel — fat-tailed
# residual robustness on the Column 6 design.
#
# API locked per plan Rev 2: ``statsmodels.miscmodels.tmodel
# .TLinearModel``. Same (y, X) design matrix as §3.1 Column 6 (six RHS
# regressors + constant). Refit runs unconditionally — no gate on §4
# diagnostic outcomes. Reports ν̂ and a side-by-side coefficient table
# vs Gaussian OLS.
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.miscmodels.tmodel import TLinearModel

# Reconstruct the Column 6 design matrix — same regressor set as §3.1
# Column 6, same ``sm.add_constant`` prefix.
_COLUMN6_REGRESSORS = [
    "cpi_surprise_ar1",
    "us_cpi_surprise",
    "banrep_rate_surprise",
    "vix_avg",
    "intervention_dummy",
    "oil_return",
]
_y_t = weekly["rv_cuberoot"]
_X_t = sm.add_constant(weekly[_COLUMN6_REGRESSORS])

# Fit Student-t likelihood. TLinearModel's fit routine estimates
# ν (degrees of freedom) jointly with β and σ via MLE on the t
# density. Standard-errors come from the observed-information matrix.
_tfit = TLinearModel(_y_t, _X_t).fit(disp=False)

# Extract ν̂ and σ̂. Per statsmodels TLinearModel source, params
# layout is [β_const, β_1, ..., β_6, df, scale]. k_extra = 2.
# param_names confirms the 'df' + 'scale' trailing labels. The
# values are NOT log-transformed — df is the Student-t degrees of
# freedom directly, scale is the t-distribution scale directly.
nu_hat = float(_tfit.params[-2])
_scale_hat = float(abs(_tfit.params[-1]))

# Build side-by-side coefficient table. Gaussian coefficients come
# from column6_fit (§3.1). Student-t coefficients are params[0:k_beta]
# where k_beta = len(_COLUMN6_REGRESSORS) + 1 (constant + six).
_k_beta = len(_COLUMN6_REGRESSORS) + 1
_gaussian_params = column6_fit.params
_gaussian_bse = column6_fit.bse
_t_params_beta = _tfit.params[:_k_beta]
_t_bse_beta = _tfit.bse[:_k_beta]

_coef_rows = []
_regressor_order = ["const"] + _COLUMN6_REGRESSORS
for _i, _name in enumerate(_regressor_order):
    _coef_rows.append(
        {
            "regressor": _name,
            "beta_gaussian": float(_gaussian_params[_name]),
            "se_gaussian_hac4": float(_gaussian_bse[_name]),
            "beta_tstudent": float(_t_params_beta[_i]),
            "se_tstudent": float(_t_bse_beta[_i]),
        }
    )
_side_by_side = pd.DataFrame(_coef_rows)

print(f"ν̂ (Student-t degrees of freedom) = {nu_hat:.4f}")
print(f"σ̂ (Student-t scale)              = {_scale_hat:.6f}")
print()
print("Side-by-side coefficient table — Gaussian OLS vs Student-t MLE:")
print(_side_by_side.to_string(index=False, float_format=lambda x: f"{x:+.6f}"))
print()
_beta_cpi_gauss = float(_gaussian_params["cpi_surprise_ar1"])
_beta_cpi_tstudent = float(_t_params_beta[_regressor_order.index("cpi_surprise_ar1")])
_se_cpi_tstudent = float(_t_bse_beta[_regressor_order.index("cpi_surprise_ar1")])
print(f"β̂_CPI Gaussian (HAC(4)): {_beta_cpi_gauss:+.6f} "
      f"(SE {float(_gaussian_bse['cpi_surprise_ar1']):.6f})")
print(f"β̂_CPI Student-t (MLE) : {_beta_cpi_tstudent:+.6f} "
      f"(SE {_se_cpi_tstudent:.6f})")
_sign_match = (
    (_beta_cpi_gauss > 0 and _beta_cpi_tstudent > 0)
    or (_beta_cpi_gauss < 0 and _beta_cpi_tstudent < 0)
)
print(f"Sign match (Gaussian vs Student-t): "
      f"{'YES' if _sign_match else 'NO'}")


**Interpretation — §5.1.** The Student-t MLE refit reports $\hat{\nu}$ (degrees of freedom), $\hat{\sigma}$ (scale), and a side-by-side $\hat{\beta}$ comparison against Gaussian OLS for each of the six Column-6 RHS regressors plus the constant.

The $\hat{\nu}$ value is the quantitative fat-tail index: Gaussian is recovered as $\hat{\nu} \to \infty$; values in the 3-5 range signal financial-return-like heavy tails; values in the 10-30 range signal mildly heavy tails that behave similarly to Gaussian except in the extreme quantiles. Whatever $\hat{\nu}$ the MLE lands on is a finding in itself — it tells us, quantitatively, *how* fat the residuals of Column 6 are, and it becomes a simulator calibration number (Task 22's reconciliation table passes $\hat{\nu}$ to the insurance engine's monte-carlo residual-shock distribution).

The side-by-side coefficient table is the comparison artifact. For each of the seven rows (constant + six RHS regressors), we show:

- $\hat{\beta}_{\text{Gaussian}}$ and its HAC(4) standard error — exactly the Column 6 output from §3.1.
- $\hat{\beta}_{\text{Student-}t}$ and its MLE-observed-information standard error — the new fit.

What to look for in the comparison:

- **Sign agreement on $\hat{\beta}_{\mathrm{CPI}}$ between the two likelihoods.** The Gaussian OLS point estimate for Column 6 from §3.1 was $\hat{\beta}_{\mathrm{CPI}}^{\text{Gauss}} = -0.000685$. If the Student-t sign matches (printed above as YES/NO), the fat-tail-robust likelihood agrees with Gaussian OLS on the direction of the effect — CPI surprise *lowers* realized vol in the following week — and the Column 6 sign is not an artifact of outlier observations pulling the regression line. If the sign disagrees, the Gaussian fit is being driven by tail observations that the Student-t downweights, and the reconciliation table must carry both signs side by side with no attempt to pick a winner.

- **Magnitude change on $\hat{\beta}_{\mathrm{CPI}}$.** A large magnitude shift between Gaussian and Student-t suggests the regression line is sensitive to extreme residuals — down-weighting them materially moves the estimated slope. A small magnitude shift means the regression is robust across likelihoods.

- **SE comparison on $\hat{\beta}_{\mathrm{CPI}}$.** The Gaussian SE is HAC(4) (robust to heteroskedasticity + low-order autocorrelation, not robust to non-normality of residuals in small samples). The Student-t SE is observed-information (exact MLE under the correct likelihood if t-distributed residuals is the true DGP). Which is tighter depends on the data — there is no theoretical prediction ex ante.

This section does **not** recommend replacing Column 6 with the Student-t fit. Column 6 is the pre-registered Gaussian primary; §5 is a robustness companion; both enter the Task 22 reconciliation table with no winner-picking. The decision for which to carry forward to simulation is made under the structural-econometrics spec (Rev 4), not under this notebook's local interpretation.